In [1]:
import pandas as pd
import phunk

import requests
import io

from rock import initialize

import numpy as np

import matplotlib.pyplot as plt


/home/kxenos/anaconda3/envs/phunk-dev/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
target = "Hektor"

# get data for target
r = requests.post(
    'https://api.ztf.fink-portal.org/api/v1/sso',
    json={
        'n_or_d': f'{target}',
        'withEphem': True,
        'withResiduals': True,
        'output-format': 'json'
    }
)

# Format output in a DataFrame
data = pd.read_json(io.BytesIO(r.content))

In [3]:
pc = phunk.PhaseCurve(
    target=target,
    epoch=data["Date"],
    phase=data["Phase"],
    mag=data["i:magpsf_red"],
    mag_err=data["i:sigmapsf"],
    band=data["i:fid"],
)
pc.get_ephems()
pc.epoch = pc.epoch_ltc


Querying ephemerides via IMCCE Miriade..


In [4]:
p0, metadata = initialize(pc, weights=pc.mag_err, remap=True, metadata=True)
pc.fit(models=["SOCCA"], p0=p0, weights=pc.mag_err, remap=True)